# Porcentaje de áreas protegidas con planes de manejo en proceso de elaboración o aprobados y/o en implementación

Este indicador nacional mide la proporción de las áreas protegidas del Estado que cuentan con un instrumento formal de administración (plan de manejo), ya sea en fase de diseño, oficialmente aprobado o en etapa de ejecución. Su objetivo es evaluar el progreso directo hacia la Meta Nacional I.2, la cual exige que al 2030 el 100% del Sistema Nacional de Áreas Protegidas cuente con estos planes. El plan de manejo es la herramienta de gobernanza habilitante fundamental; sin él, es imposible definir la zonificación, los objetos de conservación y las regulaciones de uso necesarias para transitar hacia una "gestión efectiva" del territorio.
El acceso a la planilla de datos del indicador se encuentra disponible [aquí](https://docs.google.com/spreadsheets/d/1KZgjrnVg_FQpt9kkxdm68pldOq0h7eqA/edit?usp=drive_link&ouid=103438392959464183742&rtpof=true&sd=true).

## Metodología de Cálculo
El cálculo corresponde a un cociente porcentual simple. El numerador es el número total de áreas protegidas del Estado que registran un plan de manejo (en cualquiera de las tres fases mencionadas), y el denominador es el universo total de áreas protegidas registradas oficialmente en el sistema nacional. Para otorgar mayor resolución analítica, los datos se desagregan según la designación legal o categoría de manejo del área (p. ej., Parque Nacional, Santuario de la Naturaleza, etc.), permitiendo identificar dónde se concentran los cuellos de botella administrativos.

# Capa 1: Observación y recolección
## Fuentes de Datos Utilizadas 
La información oficial para la cuantificación de este indicador se extrae del Sistema de Información de Biodiversidad (SIMBIO), administrado por el Ministerio del Medio Ambiente (MMA). Los datos específicos para este reporte fueron consolidados a partir de la planilla oficial de Áreas Protegidas del visor (disponible y descargada desde https://simbio.mma.gob.cl/CbaAP).


In [29]:
# ============================================================
# 0) Setup
# ============================================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

import io
import time
import requests

# Fuente oficial (Descargar -> Catálogo extendido)
EXCEL_EXTENDIDO_URL = "https://simbio.mma.gob.cl/CbaAP/GetExcelExtendido"

KEY = "Código AP"

In [30]:
# ------------------------------------------------------------
# Helper: descarga robusta + lectura (Excel o CSV desde URL)
# ------------------------------------------------------------
def fetch_bytes(
    url: str, timeout: int = 120, retries: int = 3, sleep_s: float = 2.0
) -> bytes:
    headers = {
        # User-Agent ayuda con algunos servidores que bloquean requests "vacíos"
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
    }
    last_err = None
    for i in range(retries):
        try:
            r = requests.get(
                url, headers=headers, timeout=timeout, allow_redirects=True
            )
            r.raise_for_status()
            return r.content
        except Exception as e:
            last_err = e
            if i < retries - 1:
                time.sleep(sleep_s)
    raise RuntimeError(
        f"No se pudo descargar {url} tras {retries} intentos. Último error: {last_err}"
    )


def read_table_from_url(url: str):
    data = fetch_bytes(url)
    # Intentar Excel primero (el endpoint se llama GetExcelExtendido)
    try:
        xls = pd.ExcelFile(io.BytesIO(data))
        sheets = xls.sheet_names
        if len(sheets) == 1:
            return pd.read_excel(xls, sheet_name=sheets[0])
        return {name: pd.read_excel(xls, sheet_name=name) for name in sheets}
    except Exception:
        # Fallback: intentar como CSV con separador inferido y encoding típico
        for enc in ("utf-8", "latin-1"):
            try:
                return pd.read_csv(
                    io.BytesIO(data), sep=None, engine="python", encoding=enc
                )
            except UnicodeDecodeError:
                continue
        return pd.read_csv(io.BytesIO(data), encoding="latin-1")


def pick_sheet(dfs: dict, include: str):
    """Selecciona una hoja por substring (case-insensitive)."""
    include = include.lower()
    for k in dfs.keys():
        if include in k.lower():
            return dfs[k], k
    # fallback: primera hoja
    first = next(iter(dfs.items()))
    return first[1], first[0]

In [31]:
# ============================================================
# 1) Lectura desde origen web SIMBIO
# ============================================================
obj = read_table_from_url(EXCEL_EXTENDIDO_URL)

# Si viene como Excel con múltiples hojas, intentamos mapear a general/manejo
if isinstance(obj, dict):
    df_general, sh_general = pick_sheet(obj, "General")
    df_manejo, sh_manejo = pick_sheet(obj, "Instrumentos de Manejo")
    df_ambiente, sh_ambiente = pick_sheet(obj, "Ambiente")
    print(f"Hojas detectadas: {list(obj.keys())}")
    print(f"Usando hoja GENERAL: {sh_general}")
    print(f"Usando hoja MANEJO : {sh_manejo}")
else:
    # Si viene en una sola tabla, la dejamos como "general" y "manejo" queda vacío
    df_general = obj
    df_manejo = pd.DataFrame()
    df_ambiente = pd.DataFrame()
    print("Se descargó una sola tabla (sin múltiples hojas).")

print("\nGeneral:", df_general.shape)
print("Manejo :", df_manejo.shape)

print("\nColumnas general:")
print(df_general.columns.tolist())

print("\nColumnas manejo:")
print(df_manejo.columns.tolist())

Hojas detectadas: ['General', 'Categoría de manejo (UICN)', 'Instrumentos de Manejo', 'Ambientes', 'Tipo Ecorregión', 'DPA', 'Cuencas', 'Historia legal', 'Ecosistemas terrestres', 'Ecosistemas marinos', 'Humedales', 'Especies', 'Servicios ecositémicos', 'Autoridad Usuario', 'Propiedad', 'Objetos de protección', 'Objetivos de protección', 'Amenaza o fuente de presiones', 'Uso de suelo', 'Planes RECOGE', 'Restauración ecológica']
Usando hoja GENERAL: General
Usando hoja MANEJO : Instrumentos de Manejo

General: (249, 17)
Manejo : (249, 5)

Columnas general:
['Código AP', 'Nombre original', 'Descripción', 'Categoría o designación', 'Tipo designación', 'Importancia', 'Superficie oficial', 'Superficie calculada', 'Fuente de información usada para indicar la superficie oficial', 'Año de inicio de la protección oficial', 'Estatus', 'Año estatus', 'Ambiente de gestión', 'Descripción objetos de protección', 'Descripción objetivos de protección', 'Descripción valores complementarios', 'Coordenad

In [32]:
# ============================================================
# 1) Validación y estandarización del KEY
# ============================================================


def normalize_key(series: pd.Series) -> pd.Series:
    # convierte a string, strip, y normaliza espacios raros
    s = series.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    # si hay cosas tipo 'nan' como texto, pásalas a NaN real
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s


for df, name in [(df_general, "general"), (df_manejo, "manejo")]:
    if KEY not in df.columns:
        raise KeyError(
            f"No encontré la columna '{KEY}' en {name}. Columnas: {df.columns.tolist()}"
        )
    df[KEY] = normalize_key(df[KEY])

# Duplicados por KEY (importante: manejo podría traer 1:N; lo veremos)
dup_gen = df_general[df_general.duplicated(KEY, keep=False)].sort_values(KEY)
dup_man = df_manejo[df_manejo.duplicated(KEY, keep=False)].sort_values(KEY)

print("Duplicados en general:", len(dup_gen))
print("Duplicados en manejo :", len(dup_man))

# Si quieres inspeccionar duplicados:
# display(dup_gen.head(20))
# display(dup_man.head(20))

Duplicados en general: 0
Duplicados en manejo : 145


In [33]:
# ============================================================
# Filtrar df_manejo: solo Plan de Manejo o No Tiene
# ============================================================

# Normalizamos la columna Tipo por seguridad
df_manejo["Tipo"] = df_manejo["Tipo"].astype(str).str.strip()

valores_validos = ["Plan de manejo", "No Tiene"]

df_manejo_filtrado = df_manejo[df_manejo["Tipo"].isin(valores_validos)].copy()

# print("Registros manejo (original):", len(df_manejo))
# print("Registros manejo (filtrado):", len(df_manejo_filtrado))

# Revisión rápida
# df_manejo_filtrado["Tipo"].value_counts()

tabla_resumen = (
    df_manejo_filtrado.groupby("Tipo", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

# Calcular porcentaje
total = tabla_resumen["n"].sum()
tabla_resumen["porcentaje"] = (tabla_resumen["n"] / total * 100).round(1)

tabla_resumen

,Tipo,n,porcentaje
0,No Tiene,36,24.7
1,Plan de manejo,110,75.3


In [34]:
# Agregar a nivel Código AP
manejo_agg = df_manejo_filtrado.groupby(KEY, as_index=False).agg(
    tiene_plan=("Tipo", lambda x: (x == "Plan de manejo").any()),
    tipo_plan=(
        "Tipo",
        lambda x: "Plan de manejo" if (x == "Plan de manejo").any() else "No Tiene",
    ),
)

manejo_agg.head()

,Código AP,tiene_plan,tipo_plan
0,WDPA-001,True,Plan de manejo
1,WDPA-002,False,No Tiene
2,WDPA-003,True,Plan de manejo
3,WDPA-004,True,Plan de manejo
4,WDPA-005,True,Plan de manejo


# Capa 2: Análisis y síntesis

Tablas intermedias

In [35]:
manejo_agg = df_manejo_filtrado.groupby(KEY, as_index=False).agg(
    tiene_plan=("Tipo", lambda x: (x == "Plan de manejo").any()),
    tipo_plan=(
        "Tipo",
        lambda x: "Plan de manejo" if (x == "Plan de manejo").any() else "No Tiene",
    ),
    anio_ultimo=("Año", "max"),
    anio_primero=("Año", "min"),
)

manejo_agg.head()

,Código AP,tiene_plan,tipo_plan,anio_ultimo,anio_primero
0,WDPA-001,True,Plan de manejo,2022,2022
1,WDPA-002,False,No Tiene,0,0
2,WDPA-003,True,Plan de manejo,2025,2025
3,WDPA-004,True,Plan de manejo,2025,2025
4,WDPA-005,True,Plan de manejo,2025,2025


In [36]:
df = df_general.merge(manejo_agg, on=KEY, how="left").merge(
    df_ambiente, on=KEY, how="left"
)

# Si no aparece en manejo → No Tiene
df["tiene_plan"] = df["tiene_plan"].fillna(False)
df["tipo_plan"] = df["tipo_plan"].fillna("No Tiene")

df.head()

/var/folders/ns/7cctx66n6f9330x52d1c4zwm0000gn/T/ipykernel_10904/886645857.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["tiene_plan"] = df["tiene_plan"].fillna(False)


,Código AP,Nombre original,Descripción,Categoría o designación,Tipo designación,Importancia,Superficie oficial,Superficie calculada,Fuente de información usada para indicar la superficie oficial,Año de inicio de la protección oficial,Estatus,Año estatus,Ambiente de gestión,Descripción objetos de protección,Descripción objetivos de protección,Descripción valores complementarios,Coordenada geográfica,tiene_plan,tipo_plan,anio_ultimo,anio_primero,Presencia,Procentaje del AP
0,WDPA-011,La Portada,"La unidad destaca por “La Portada”, una secuen...",Monumento Natural,Áreas Protegidas,Área conocida a nivel nacional e internacional...,31.0,NaN,Decreto de creación,1990,Designada,1990,NaN,Gran arco de La Portada (acantilado marino fós...,Preservar un rasgo geomorfológico de carácter ...,NaN,NaN,True,Plan de manejo,1995.0,1995.0,Terrestre,0.0
1,WDPA-015,Cerro Ñielol,El Monumento se ubica en el sector más austral...,Monumento Natural,Áreas Protegidas,Forma parte de uno de los sitios más australes...,90.0,NaN,Decreto de creación,1939,Designada,1987,NaN,Especies forestales (Decreto de 1949). Bosque ...,Conservar especies forestales naturales (decre...,NaN,NaN,True,Plan de manejo,2008.0,2008.0,Terrestre,0.0
2,WDPA-019,Cinco Hermanas,"El Monumento Natural Cinco Hermanas, correspon...",Monumento Natural,Áreas Protegidas,Se compone de seis pequeñas islas. La vegetaci...,228.0,NaN,Decreto de creación,1964,Designada,1982,NaN,"Flora, la Fauna y las Bellezas Escénicas (decr...",Proteger el bosque especialmente la formación ...,NaN,NaN,True,Plan de manejo,2014.0,2014.0,Terrestre,0.0
3,WDPA-016,Contulmo,"Esta área, destaca por la protección de las es...",Monumento Natural,Áreas Protegidas,Forma parte de la cordillera de Nahuelbuta y e...,82.0,NaN,Decreto de creación,1941,Designada,1982,NaN,Bosques relictos de altitudes bajas de la Cord...,"Conservación de flora y fauna (Plan de manejo,...",NaN,NaN,True,Plan de manejo,2000.0,2000.0,Terrestre,0.0
4,WDPA-023,Cueva del Milodón,Enorme caverna de más de 200 metros de profund...,Monumento Natural,Áreas Protegidas,En esta unidad podemos encontrar el bosque mag...,189.0,NaN,Decreto de creación,1993,Designada,1993,NaN,Recursos arqueológicos y geomorfológicos que e...,Conservar y proteger los importantes recursos ...,NaN,NaN,True,Plan de manejo,1998.0,1998.0,Terrestre,0.0


In [37]:
df["Presencia"].value_counts(dropna=False)

Presencia
Terrestre    172
Marino        49
Acuático      38
Name: count, dtype: int64

# Capa 3: Repporte Digital

## Resultados

A la fecha de reporte, la evaluación del sistema arroja que 109 de las 145 áreas protegidas registradas (75,3%) cuentan actualmente con un Plan de Manejo, evidenciando un progreso sustantivo en la formalización de la conservación a nivel nacional. No obstante, la desagregación de los datos revela que la distribución de este avance es altamente heterogénea según la categoría de protección:
Alta cobertura: Las Reservas Forestales alcanzan un 100% (n=2); las Reservas Nacionales un 82,2%; los Monumentos Naturales un 75,0%; y las Áreas de Conservación de Múltiples Usos un 73,3%.
Cobertura media-alta: Los Parques Nacionales —la categoría de mayor relevancia estricta dentro del sistema— alcanzan un 62,1%, reflejando un avance importante pero con un margen de formalización aún pendiente.
Baja o nula cobertura: Los Santuarios de la Naturaleza evidencian una brecha crítica, con solo un 8,8% de sus áreas amparadas por un plan de manejo. Por su parte, las Reservas Marinas evaluadas no registran planes de manejo vigentes (0%).

In [38]:
# Área total de Chile continental (ha) extraído desde indicaddor A.1
AREA_CHILE_CONTINENTAL = 75688051.58421065

# Asegurar tipos
df["tiene_plan"] = df["tiene_plan"].astype(bool)
df["Superficie oficial"] = pd.to_numeric(df["Superficie oficial"], errors="coerce")

# Valores de Presencia a considerar
PRESENCIAS_VALIDAS = ["Terrestre", "Acuático"]

# 1) Suma de superficie oficial con plan y presencia válida
superficie_con_plan = df.loc[
    (df["tiene_plan"]) & (df["Presencia"].isin(PRESENCIAS_VALIDAS)),
    "Superficie oficial",
].sum(skipna=True)

# 2) Porcentaje respecto al área continental
porcentaje_chile = (superficie_con_plan / AREA_CHILE_CONTINENTAL) * 100

# superficie_con_plan, porcentaje_chile

# ============================================================
# Tabla de resultados: superficie con plan de manejo
# ============================================================

tabla_resultados = pd.DataFrame(
    {
        "Indicador": [
            "Superficie de AP con plan de manejo (Terrestre + Acuático)",
            "Área continental de Chile",
        ],
        "Superficie (ha)": [superficie_con_plan, AREA_CHILE_CONTINENTAL],
    }
)

# Calcular porcentaje solo para la primera fila
tabla_resultados["Porcentaje del área continental (%)"] = [
    round(porcentaje_chile, 2),
    100.0,
]

tabla_resultados

,Indicador,Superficie (ha),Porcentaje del área continental (%)
0,Superficie de AP con plan de manejo (Terrestre...,8.083414e+06,10.68
1,Área continental de Chile,7.568805e+07,100.00


In [39]:
# Buscar columna de categoría/designación
[c for c in df.columns if "categor" in c.lower() or "design" in c.lower()]

['Categoría o designación', 'Tipo designación']

In [40]:
# ============================================================
# Normalizar categorías: Reserva Nacional (antiguo) → Reserva Nacional
# ============================================================
CAT_COL = "Categoría o designación"

df[CAT_COL] = (
    df[CAT_COL]
    .astype(str)
    .str.strip()
    .replace({"Reserva Nacional (antiguo)": "Reserva Nacional"})
)

# Verificación rápida
df[CAT_COL].value_counts().loc[
    lambda s: s.index.str.contains("Reserva Nacional", case=False)
]

Categoría o designación
Reserva Nacional    49
Name: count, dtype: int64

In [41]:
tabla_conteo = (
    df.groupby([CAT_COL, "tipo_plan"], dropna=False).size().reset_index(name="n")
)

totales = (
    tabla_conteo.groupby(CAT_COL, as_index=False)["n"]
    .sum()
    .rename(columns={"n": "total_categoria"})
)

tabla_pct = (
    tabla_conteo.merge(totales, on=CAT_COL, how="left")
    .assign(porcentaje=lambda d: 100 * d["n"] / d["total_categoria"])
    .sort_values([CAT_COL, "tipo_plan"])
)

tabla_pct

,Categoría o designación,tipo_plan,n,total_categoria,porcentaje
0,Monumento Natural,No Tiene,6,21,28.571429
1,Monumento Natural,Plan de manejo,15,21,71.428571
2,Parque Nacional,No Tiene,22,58,37.931034
3,Parque Nacional,Plan de manejo,36,58,62.068966
4,Reserva Forestal,Plan de manejo,2,2,100.000000
5,Reserva Marina,No Tiene,6,6,100.000000
6,Reserva Nacional,No Tiene,11,49,22.448980
7,Reserva Nacional,Plan de manejo,38,49,77.551020
8,Santuario de la Naturaleza,No Tiene,99,108,91.666667
9,Santuario de la Naturaleza,Plan de manejo,9,108,8.333333


In [42]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# Tabla base: conteos por categoría y tipo de plan
# ============================================================

tabla_n = df.groupby([CAT_COL, "tipo_plan"]).size().reset_index(name="n")

tabla_n

,Categoría o designación,tipo_plan,n
0,Monumento Natural,No Tiene,6
1,Monumento Natural,Plan de manejo,15
2,Parque Nacional,No Tiene,22
3,Parque Nacional,Plan de manejo,36
4,Reserva Forestal,Plan de manejo,2
5,Reserva Marina,No Tiene,6
6,Reserva Nacional,No Tiene,11
7,Reserva Nacional,Plan de manejo,38
8,Santuario de la Naturaleza,No Tiene,99
9,Santuario de la Naturaleza,Plan de manejo,9


In [43]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Define specs: Row 1, Col 1 spans 2 rows.
# The slot directly below it (Row 2, Col 1) must be None.
specs = [
    [{"type": "domain", "rowspan": 2}, {"type": "domain"}],  # Row 1
    [None, {"type": "domain"}],  # Row 2 (merged into above)
    [{"type": "domain"}, {"type": "domain"}],  # Row 3
    [{"type": "domain"}, {"type": "domain"}],  # Row 4
]
titles = [
    "Reserva Nacional",
    "Monumento Natural",
    "Área de Conservación de Múltiples Usos",
    "Santuario de la Naturaleza",
    "Reserva Forestal",
    "Reserva Marina",
]
fig = make_subplots(
    rows=4,
    cols=2,
    specs=specs,
    subplot_titles=[
        "Parque Nacional",
        "Reserva Nacional",
        "Monumento Natural",
        "Área de Conservación de Múltiples Usos",
        "Santuario de la Naturaleza",
        "Reserva Forestal",
        "Reserva Marina",
    ],
)

# 2. Add the "merged" tall chart (addresses the top-left corner)
data = tabla_n[tabla_n[CAT_COL] == "Parque Nacional"]
fig.add_trace(
    go.Pie(
        labels=data["tipo_plan"],
        values=data["n"],
        name="Large",
        marker=dict(colors=["#4E79A7", "#F28E2B"]),
    ),
    row=1,
    col=1,
)

# 3. Add the remaining 6 charts
# Note: We manually place these since the grid logic is now asymmetrical
positions = [(1, 2), (2, 2), (3, 1), (3, 2), (4, 1), (4, 2)]

for i, (r, c) in enumerate(positions):
    data = tabla_n[tabla_n[CAT_COL] == titles[i]]
    fig.add_trace(
        go.Pie(
            labels=data["tipo_plan"],
            values=data["n"],
            marker=dict(colors=["#4E79A7", "#F28E2B"]),
        ),
        row=r,
        col=c,
    )

fig.update_layout(height=1000, showlegend=True)
fig.show()

In [46]:
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Asegurar que el año sea numérico
df["anio_ultimo"] = pd.to_numeric(df["anio_ultimo"], errors="coerce")

# Filtrar solo áreas que tienen plan válido
df_plot = df.loc[(df["tiene_plan"] == True) & (df["anio_ultimo"] > 0)].copy()

import plotly.graph_objects as go

fig = go.Figure(data=[go.Histogram(x=df_plot["anio_ultimo"])])
fig.update_layout(
    title="Distribución de los Años de Publicación del Último Plan de Manejo"
)
fig.write_image("histograma_anio_ultimo_plan.png")

fig.show()

In [47]:
import plotly.express as px

df_plot["decada"] = (df_plot["anio_ultimo"] // 10) * 10
conteo_decadas = df_plot["decada"].value_counts().sort_index()

fig = px.bar(
    x=conteo_decadas.index.astype(str),
    y=conteo_decadas.values,
    labels={"x": "Década", "y": "Número de Áreas Protegidas"},
    title="Planes de manejo por década",
)
fig.show()

In [48]:
anio_actual = datetime.now().year
df_plot["antiguedad"] = anio_actual - df_plot["anio_ultimo"]

df_plot["antiguedad"].describe()

count    111.000000
mean      16.738739
std       10.365776
min        0.000000
25%        8.000000
50%       13.000000
75%       27.000000
max       38.000000
Name: antiguedad, dtype: float64

## Observaciones

Alcanzar una cobertura basal del 75,3% a nivel de sistema representa un hito administrativo robusto. Este piso resulta fundamental de cara a la implementación del nuevo Servicio de Biodiversidad y Áreas Protegidas (SBAP, Ley N° 21.600), ya que entrega a la nueva institucionalidad una red que, en su gran mayoría, ya cuenta con instrumentos rectores sobre los cuales auditar y planificar la gestión territorial.

## Brechas

Si bien el indicador demuestra un avance cuantitativo favorable, su diseño metodológico actual presenta limitaciones que condicionan la interpretación del cumplimiento real de la meta:
- Carencia de parámetros sobre obsolescencia y vigencia ecológica: El diseño actual del indicador cuantifica la existencia de un plan de manejo desde una perspectiva nominal (presencia/ausencia), sin incorporar variables relacionadas con la vigencia temporal de dichos instrumentos. De acuerdo con los registros, una proporción significativa de las áreas protegidas contabilizadas de forma positiva posee planes formulados hace más de 25 años. Al ponderar con igual valor estadístico un instrumento obsoleto frente a uno actualizado bajo criterios contemporáneos (p. ej., adaptación al cambio climático y participación local), la métrica tiende a sobredimensionar el estado de la "gestión efectiva" del país.
- Desafío estadístico por homologación normativa (Ley N° 21.600): La próxima implementación del Artículo 56 de la Ley N° 21.600 que crea el Servicio de Biodiversidad y Áreas Protegidas y el Sistema Nacional de Áreas Protegidas, obligará a homologar las actuales categorías de protección (tales como Santuarios de la Naturaleza y Reservas Marinas, que hoy presentan los menores índices de cumplimiento) hacia las nuevas figuras oficiales de conservación. Este proceso de reordenamiento jurídico alterará estructuralmente tanto el numerador como el denominador del indicador en el corto y mediano plazo. Por consiguiente, el sistema SIMBIO enfrenta el desafío técnico de desarrollar trazadores relacionales robustos que aseguren la continuidad y comparabilidad de las series de tiempo, evitando que la recategorización administrativa introduzca sesgos estadísticos en el reporte histórico de progreso ante el CBD.